In [ ]:
# This script has alternate functions that use named tuples instead of dictionaries

using MarineHydro
using PyCall
cpt = pyimport("capytaine")


# Create Mesh object
radius = 1.5 # 
center = (0.0,0.0,0.0)
len = 2.5
faces_max_radius = 0.5
cptmesh = cpt.meshes.predefined.mesh_horizontal_cylinder(
            radius=radius,
            center=center, 
            length=len, 
            faces_max_radius = faces_max_radius
            ).immersed_part()

# Capytaine solve
function cpt_hydro_coeffs(omega, cptmesh)

    # Description of problem
    h = Inf # sea depth [m]
    beta = 0 # incident wave angle [rad]
    translational_DOFs = ["Surge"]
    rotational_DOFs = ["Pitch"]

    # Create FloatingBody object
    cptbody = cpt.FloatingBody(mesh=cptmesh)
    foreach(dof -> cptbody.add_translation_dof(name=dof), translational_DOFs)
    foreach(dof -> cptbody.add_rotation_dof(name=dof), rotational_DOFs)
    cptbody.active_dofs = [translational_DOFs; rotational_DOFs]
    cptbody.name = "body 1"
    cptbody.center_of_mass = (0, 0, 0)
    cptbody.rotation_center = (0, 0, 0)

    # Create BEMSolver object
    solver = cpt.BEMSolver()
    # Create DiffractionProblem object
    diff_prob = cpt.DiffractionProblem(body=cptbody, wave_direction=beta, omega=omega, water_depth=h) 

    # Create list of RadiationProblem objects
    rad_prob = [] 

    for dof in cptbody.active_dofs

        prob = cpt.RadiationProblem(
            body=cptbody, 
            omega=omega, 
            radiating_dof=dof, 
            water_depth=h
        )

        push!(rad_prob, prob) # this is a Julia way of appending p to the list rad_prob
    end

    # Solve diffraction and radiation problem
    diff_result = solver.solve(diff_prob,keep_details=(true))
    rad_result = solver.solve_all(rad_prob,keep_details=(true))
    dataset = cpt.assemble_dataset([rad_result; diff_result])

    A = dropdims(dataset.added_mass.values, dims=1)
    B = dropdims(dataset.radiation_damping.values, dims=1)
    F_ex = dropdims(dataset.excitation_force.values, dims=1)'

    return A, B, F_ex
end

cpt_hydro_coeffs (generic function with 1 method)

In [33]:
cpt_hydro_coeffs(1.3, cptmesh)

[16:49:19] WARNING  The rotation dof Pitch has been initialized around the     
                    origin of the domain (0, 0, 0).                            
Solving BEM problems ---------------------------------------- 100% 0:00:00


([7247.6264854352585 -3284.485459718001; -3247.5584442497448 2906.4040953725566], [331.8258896615045 -111.67045091647408; -107.52756530677735 36.1778213519136], ComplexF64[363.9857754541422 + 23459.314992249972im; -122.49220270358842 - 7871.336361210728im;;])

In [34]:
# MarineHydro solve

# MH_solve(omega, cptmesh)

mesh = Mesh(cptmesh)


# struct FloatingBody
#     mesh::Mesh
#     dofs::Dict # {name::Str : dofs::AbstractMatrix (nfaces,3)}

#     function FloatingBody(mesh::Mesh, dofs::Dict)
#         #add assert statements

#         return new(mesh, dofs)
#     end
# end



Mesh([-1.25 -1.4623918682727355 -0.33378140093447134; -1.25 -1.4623918682727357 0.0; … ; 1.25 1.4623918682727355 -0.3337814009344717; 1.25 1.4623918682727355 -2.7755575615628914e-17], [21 33 32 12; 33 42 41 32; … ; 82 79 78 81; 59 56 55 58], [-0.9375 0.3254128043381684 -1.4257266509268143; -0.3125 0.3254128043381684 -1.4257266509268143; … ; 1.25 1.2349086887636433 -0.14092992483899916; 1.25 -1.2349086887636436 -0.14092992483899902], [0.0 0.2225209339563142 -0.9749279121818238; 0.0 0.2225209339563142 -0.9749279121818238; … ; 1.0 0.0 0.0; 1.0 0.0 0.0], [0.41722675116808966, 0.41722675116808966, 0.41722675116808966, 0.41722675116808966, 0.4172267511680895, 0.4172267511680895, 0.4172267511680895, 0.4172267511680895, 0.41722675116808955, 0.41722675116808955  …  0.054235467389694765, 0.02711773369484739, 0.02711773369484736, 0.05423546738969476, 0.05423546738969476, 0.05423546738969479, 0.08135320108454216, 0.08135320108454208, 0.13558866847423695, 0.13558866847423684], [0.45723765550288903,

In [ ]:
struct FloatingBody{T <: NamedTuple}
    mesh::Mesh
    dofs::T 


    function FloatingBody(mesh::Mesh, dofs::T) where {T <: NamedTuple}

        
        return new{T}(mesh, dofs)
    end
end

In [36]:
dofs = (
    heave = rand(mesh.nfaces, 3), 
    pitch = rand(mesh.nfaces, 3)
)
length(dofs)

2

In [37]:
floatingbody = FloatingBody(mesh,dofs)

FloatingBody{@NamedTuple{heave::Matrix{Float64}, pitch::Matrix{Float64}}}(Mesh([-1.25 -1.4623918682727355 -0.33378140093447134; -1.25 -1.4623918682727357 0.0; … ; 1.25 1.4623918682727355 -0.3337814009344717; 1.25 1.4623918682727355 -2.7755575615628914e-17], [21 33 32 12; 33 42 41 32; … ; 82 79 78 81; 59 56 55 58], [-0.9375 0.3254128043381684 -1.4257266509268143; -0.3125 0.3254128043381684 -1.4257266509268143; … ; 1.25 1.2349086887636433 -0.14092992483899916; 1.25 -1.2349086887636436 -0.14092992483899902], [0.0 0.2225209339563142 -0.9749279121818238; 0.0 0.2225209339563142 -0.9749279121818238; … ; 1.0 0.0 0.0; 1.0 0.0 0.0], [0.41722675116808966, 0.41722675116808966, 0.41722675116808966, 0.41722675116808966, 0.4172267511680895, 0.4172267511680895, 0.4172267511680895, 0.4172267511680895, 0.41722675116808955, 0.41722675116808955  …  0.054235467389694765, 0.02711773369484739, 0.02711773369484736, 0.05423546738969476, 0.05423546738969476, 0.05423546738969479, 0.08135320108454216, 0.081353201

In [38]:
function radiation_bc(floatingbody::FloatingBody, omega)
    """
    Calculates the radiation boundary conditions for floating bodies at each panel.

    # Arguments
    - `floatingbody::FloatingBody`: The floating body
    - `omega`: The frequency of the incident ocean wave ~~~.

    # Returns
    - The (Neumann) radiation boundary condition values for each panel.
"""
    normals_mat = floatingbody.mesh.normals
    bc = map(dofs) do dof_mat
        bc_value = -1im .* omega .* sum(normals_mat .* dof_mat, dims=2) 
        return bc_value
    end
    return bc
end

radiation_bc (generic function with 1 method)

In [39]:
bcs = radiation_bc(floatingbody,omega)



(heave = ComplexF64[-0.0 + 1.0517259981493814im; -0.0 + 0.9790729496808412im; … ; 0.0 - 0.697817100157131im; 0.0 - 0.6538333779746723im;;], pitch = ComplexF64[-0.0 + 0.936225641664327im; -0.0 + 0.6403296959824405im; … ; 0.0 - 0.5887113831127585im; 0.0 - 0.17476953740270298im;;])

In [40]:
pressure = ones(mesh.nfaces,1)

80×1 Matrix{Float64}:
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 ⋮
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0

In [41]:
forces = map(dofs) do dof_mat # get forces/moments in directions that correspond to dofs 
    normal_dof_amp_on_face = -sum(dof_mat .* mesh.normals, dims=2)
    force_val = sum(pressure .* normal_dof_amp_on_face .* mesh.areas)
    return force_val
end
collect(values(forces))

2-element Vector{Float64}:
 2.9874873774138475
 4.609821605456555

In [42]:
function integrate_pressure(floatingbody::FloatingBody, pressure)
  mesh = floatingbody.mesh
  forces = map(dofs) do dof_mat # get forces/moments in directions that correspond to dofs 
    normal_dof_amp_on_face = -sum(dof_mat .* mesh.normals, dims=2)
    force_val = sum(pressure .* normal_dof_amp_on_face .* mesh.areas)
    return force_val
  end
  return collect(values(forces)) # convert named tuple into a vector
end

integrate_pressure (generic function with 1 method)

In [43]:
p = integrate_pressure(floatingbody,pressure)

2-element Vector{Float64}:
 2.9874873774138475
 4.609821605456555

In [44]:
length(dofs)

2

In [54]:
function calculate_radiation_forces(floatingbody::FloatingBody, omega)
    rho = 1023
    g = 9.81
    k = omega^2 / g
    mesh = floatingbody.mesh
    S, D = assemble_matrix_wu(mesh, k)
    rad_bcs = radiation_bc(floatingbody, omega) 
    column_vectors = [ 
        begin
            rad_bc = rad_bcs[i] 
            potential = solve(D, S, rad_bc) 
            pressure = 1im * rho * omega * potential
            integrate_pressure(floatingbody, pressure)
        end 
        for i in 1:length(dofs) # loop through all dofs, get force column vector, and assemble radiation mat
    ]
    rad_mat = hcat(column_vectors...)
    A = real.(rad_mat)./omega^2
    B = imag.(rad_mat)./omega
    return A, B
end



# for rad_dof in keys(floatingbody.dofs) # radiating dofs
#     rad_bc = rad_bcs[rad_dof]
#     potential = solve(D, S, rad_bc)
#     pressure = 1im * rho * omega * potential
#     forces = integrate_pressure(floatingbody, pressure)
#     for inf_dof in keys(forces) # influenced dofs
#         force = forces[inf_dof]
#         Added_mass[(omega, inf_dof, rad_dof)] = real(force)/omega^2
#         Radiation_damping[(omega, inf_dof, rad_dof)] = imag(force)/omega 
#     end
# end  

calculate_radiation_forces (generic function with 1 method)

In [55]:
cpt_hydro_coeffs(1.3, cptmesh)

[16:56:57] WARNING  The rotation dof Pitch has been initialized around the     
                    origin of the domain (0, 0, 0).                            
Solving BEM problems ---------------------------------------- 100% 0:00:00


([7247.6264854352585 -3284.485459718001; -3247.5584442497448 2906.4040953725566], [331.8258896615045 -111.67045091647408; -107.52756530677735 36.1778213519136], ComplexF64[363.9857754541422 + 23459.314992249972im; -122.49220270358842 - 7871.336361210728im;;])

In [58]:
A, B = calculate_radiation_forces(floatingbody, 1.3)

([5597.257041126935 4925.742149843824; 4906.795469481974 5725.138954599711], [578.568268904529 852.2927258238861; 854.014857763092 1335.6450290174967])

In [59]:
println(eltype(A))

Float64


In [62]:
using Zygote
grad, = Zygote.gradient(1.3) do w
    A, B = calculate_radiation_forces(floatingbody, w)
    return A[1, 1] 
end

(613.7925660119097,)

In [61]:
A_w_jac = Zygote.jacobian(w -> real.(calculate_radiation_forces(floatingbody, w)[1]), 1.3)

InexactError: InexactError: Float64(613.7925660119097 - 342.34808810918844im)